# ML-Based Trading Strategy for Tesla, Inc.


Financial markets are complex and highly dynamic, making accurate prediction of price movements a challenging task. Traditional trading strategies often rely on technical indicators or human intuition, which may fail to adapt effectively to changing market conditions.

In recent years, machine learning has been increasingly applied to financial forecasting and algorithmic trading due to its ability to learn patterns from historical data and generate data-driven trading signals. This project explores the development of a machine learning-based trading strategy using historical stock data for Tesla, Inc. over the period 2015 to 2025.

The objective of this project is to build a predictive model that can generate buy and sell signals based on learned patterns in market data, and evaluate its performance through a simulated trading environment. The model's effectiveness will be assessed using key financial metrics such as returns, risk, and Sharpe ratio, and compared against a baseline moving average crossover strategy and a simple buy-and-hold approach.

To make the system interpretable and interactive, a Streamlit-based dashboard will be developed to visualize trading signals, portfolio performance, and model predictions in a user-friendly interface.

## 2. Data

This project uses historical daily stock market data for Tesla, Inc. covering the period from 2015 to 2025.

The dataset includes the following daily trading fields:

- Open
- High
- Low
- Close
- Adjusted Close
- Volume

Data is obtained programmatically using the Python `yfinance` library, which provides access to historical financial market data from Yahoo Finance. Daily frequency data is selected to support:

- computation of technical indicators,
- generation of trading signals,
- evaluation of machine learning-based trading strategies.

An optional extension may involve incorporating additional stocks to evaluate the robustness and generalizability of the proposed trading strategy across multiple assets and market conditions.

In [1]:
# Data Acquisition
# Data is obtained from the wall street journal website
import pandas as pd

tsla = pd.read_csv("./data/HistoricalPrices.csv")
tsla.head()


,Date,Open,High,Low,Close,Volume
0,12/31/25,456.10,456.55,449.30,449.72,49077961.0
1,12/30/25,461.09,463.12,453.83,454.43,59238461.0
2,12/29/25,469.00,469.40,459.00,459.64,66263031.0
3,12/26/25,485.23,489.09,473.82,475.19,58780660.0
4,12/24/25,488.48,490.90,476.80,485.40,41285434.0


In [2]:
tsla.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     2767 non-null   object 
 1    Open    2767 non-null   float64
 2    High    2767 non-null   float64
 3    Low     2767 non-null   float64
 4    Close   2767 non-null   float64
 5    Volume  2767 non-null   float64
dtypes: float64(5), object(1)
memory usage: 129.8+ KB


# Data Preparation and Timeline

The dataset spans from January 2015 to December 2025, encompassing a comprehensive 11-year period of Tesla's historical stock prices.

## Data Split Strategy

The data will be partitioned into two distinct periods for model development and evaluation:

| Period | Time Range | Purpose |
|--------|-----------|---------|
| **Training/Validation** | 2015–2024 | Model development and hyperparameter tuning |
| **Testing/Backtesting** | 2025 | Out-of-sample performance evaluation |

This temporal split ensures that the model is evaluated on future, unseen data, providing a realistic assessment of its predictive capability in a live trading scenario.

In [3]:
# Data splitting
tsla["Date"] = pd.to_datetime(tsla["Date"], format= "mixed", dayfirst=True)

df = tsla.sort_values("Date")

In [4]:
df.head()

,Date,Open,High,Low,Close,Volume
2705,2015-01-04,12.5800,12.8200,12.4033,12.5060,56919373.46
2684,2015-01-05,15.3293,15.4513,14.6937,15.0687,79225401.36
2664,2015-01-06,16.7606,16.7733,16.4980,16.6300,37575893.59
2642,2015-01-07,18.0740,18.1746,17.8567,17.9433,31518392.37
2599,2015-01-09,16.0227,16.4000,15.7980,15.9087,81821559.03


In [5]:
df.tail()

,Date,Open,High,Low,Close,Volume
4,2025-12-24,488.48,490.90,476.80,485.40,41285434.0
3,2025-12-26,485.23,489.09,473.82,475.19,58780660.0
2,2025-12-29,469.00,469.40,459.00,459.64,66263031.0
1,2025-12-30,461.09,463.12,453.83,454.43,59238461.0
0,2025-12-31,456.10,456.55,449.30,449.72,49077961.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2767 entries, 2705 to 0
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     2767 non-null   datetime64[ns]
 1    Open    2767 non-null   float64       
 2    High    2767 non-null   float64       
 3    Low     2767 non-null   float64       
 4    Close   2767 non-null   float64       
 5    Volume  2767 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 151.3 KB


In [8]:
# create the training dataset
training_data = df[
    (df["Date"] >= "2015-01-02") &
    (df["Date"] <= "2024-12-31")
].copy()

training_data.head()

,Date,Open,High,Low,Close,Volume
2705,2015-01-04,12.5800,12.8200,12.4033,12.5060,56919373.46
2684,2015-01-05,15.3293,15.4513,14.6937,15.0687,79225401.36
2664,2015-01-06,16.7606,16.7733,16.4980,16.6300,37575893.59
2642,2015-01-07,18.0740,18.1746,17.8567,17.9433,31518392.37
2599,2015-01-09,16.0227,16.4000,15.7980,15.9087,81821559.03


In [12]:
df.shape

(2767, 6)

In [10]:
training_data.shape

(2517, 6)

In [13]:
# create the testing dataset
test_data = df[
    (df["Date"] >= "2025-01-02") &
    (df["Date"] <= "2025-12-31")
].copy()
test_data.head()

,Date,Open,High,Low,Close,Volume
189,2025-01-04,263.800,277.4500,259.25,268.46,146486891.0
168,2025-01-05,280.010,290.8688,279.81,280.52,99658969.0
127,2025-01-07,298.460,305.8900,293.21,300.71,145085703.0
105,2025-01-08,306.205,309.3100,297.82,302.63,89121445.0
63,2025-01-10,443.800,462.2900,440.75,459.46,98122289.0


In [14]:
test_data.shape

(250, 6)

In [15]:
training_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)

In [16]:
# Verify the new indices
training_data.head()

,Date,Open,High,Low,Close,Volume
0,2015-01-04,12.5800,12.8200,12.4033,12.5060,56919373.46
1,2015-01-05,15.3293,15.4513,14.6937,15.0687,79225401.36
2,2015-01-06,16.7606,16.7733,16.4980,16.6300,37575893.59
3,2015-01-07,18.0740,18.1746,17.8567,17.9433,31518392.37
4,2015-01-09,16.0227,16.4000,15.7980,15.9087,81821559.03


In [17]:
test_data.head()

,Date,Open,High,Low,Close,Volume
0,2025-01-04,263.800,277.4500,259.25,268.46,146486891.0
1,2025-01-05,280.010,290.8688,279.81,280.52,99658969.0
2,2025-01-07,298.460,305.8900,293.21,300.71,145085703.0
3,2025-01-08,306.205,309.3100,297.82,302.63,89121445.0
4,2025-01-10,443.800,462.2900,440.75,459.46,98122289.0


In [18]:
# Date Range
print("\nTraining Data Date Range:")
print(training_data["Date"].min(), "to", training_data["Date"].max())
print("\nTesting Data Date Range:")
print(test_data["Date"].min(), "to", test_data["Date"].max())


Training Data Date Range:
2015-01-04 00:00:00 to 2024-12-31 00:00:00

Testing Data Date Range:
2025-01-04 00:00:00 to 2025-12-31 00:00:00


In [19]:
# export training and test dataframes to csv files
training_data.to_csv("./data/training_data.csv", index=False)
test_data.to_csv("./data/test_data.csv", index=False)

# Exploratory Data Analysis